# Fleet Strategy Engine EDA

Use this notebook to inspect the input batch and, after running the CLI, verify the recommendation output.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 180)

INPUT_PATH = Path('data/sample_data.csv')
OUTPUT_PATH = Path('outputs/recommendations.csv')
SUMMARY_PATH = Path('outputs/summary.json')

raw = pd.read_csv(INPUT_PATH)
recommendations = pd.read_csv(OUTPUT_PATH) if OUTPUT_PATH.exists() else None

raw.shape, None if recommendations is None else recommendations.shape

## Input Schema And Quality Checks

In [ ]:
raw.head()

In [ ]:
raw.info()

In [ ]:
quality_checks = {
    'row_count': len(raw),
    'station_count': raw['station'].nunique(),
    'segment_count': raw['segment'].nunique(),
    'blank_cells': int(raw.isna().sum().sum()),
    'duplicate_station_segment_rows': int(raw.duplicated(['station', 'segment']).sum()),
    'invalid_utilization_rows': int(((raw['utilization_pct'] < 0) | (raw['utilization_pct'] > 100)).sum()),
    'invalid_market_share_rows': int(((raw['market_share_pct'] < 0) | (raw['market_share_pct'] > 100)).sum()),
    'non_positive_fleet_rows': int((raw['fleet_size'] <= 0).sum()),
    'non_positive_competitor_rate_rows': int((raw['competitor_rate'] <= 0).sum()),
}
quality_checks

## Input Distributions

In [ ]:
raw.describe(include='all')

In [ ]:
raw.groupby('segment').agg(
    rows=('segment', 'size'),
    avg_fleet_size=('fleet_size', 'mean'),
    avg_utilization=('utilization_pct', 'mean'),
    avg_rate=('avg_daily_rate', 'mean'),
    avg_competitor_rate=('competitor_rate', 'mean'),
    avg_market_share=('market_share_pct', 'mean'),
).round(2).sort_values('rows', ascending=False)

In [ ]:
input_scan = raw.assign(
    daily_margin=lambda df: df['avg_daily_rate'] - df['avg_daily_fleet_cost'] - df['avg_daily_operating_cost'],
    price_gap_pct=lambda df: (df['avg_daily_rate'] - df['competitor_rate']) / df['competitor_rate'] * 100,
)

input_scan[['fleet_size', 'utilization_pct', 'daily_margin', 'price_gap_pct', 'market_share_pct']].hist(figsize=(12, 8), bins=20);

## Useful Pre-Recommendation Queries

In [ ]:
# Capacity-constrained rows where demand may exceed available fleet.
input_scan.query('utilization_pct >= 90').sort_values(['utilization_pct', 'daily_margin'], ascending=False).head(20)

In [ ]:
# Potential overfleet rows.
input_scan.query('utilization_pct < 75').sort_values(['utilization_pct', 'market_share_pct']).head(20)

In [ ]:
# High utilization that may be driven by discounting versus competitor.
input_scan.query('utilization_pct >= 90 and price_gap_pct <= -10').sort_values('price_gap_pct').head(20)

In [ ]:
# Margin risk rows.
input_scan.query('daily_margin <= 0').sort_values('daily_margin')

## Recommendation Output Checks

Run the CLI first if `recommendations` is `None`:

```bash
uv run python main.py --input data/sample_data.csv --output outputs/recommendations.csv --summary-output outputs/summary.json
```

In [ ]:
if recommendations is None:
    print('No output file found yet. Run the CLI command above, then restart or rerun the first cell.')
else:
    display(recommendations.head())

In [ ]:
if recommendations is not None:
    display(recommendations['recommendation'].value_counts().reindex(['BUY', 'HOLD', 'REDUCE'], fill_value=0))
    display(pd.crosstab(recommendations['segment'], recommendations['recommendation']))

In [ ]:
if recommendations is not None:
    display(recommendations.groupby('recommendation').agg(
        rows=('recommendation', 'size'),
        avg_utilization=('utilization_pct', 'mean'),
        avg_margin=('daily_margin', 'mean'),
        avg_market_share=('market_share_pct', 'mean'),
        net_delta=('recommended_fleet_delta', 'sum'),
    ).round(2))

## Recommendation Verification Queries

In [ ]:
# BUY candidates ranked by recommended fleet increase.
if recommendations is not None:
    display(recommendations.query("recommendation == 'BUY'").sort_values(['recommended_fleet_delta', 'utilization_pct'], ascending=False).head(20))

In [ ]:
# REDUCE candidates ranked by recommended reduction.
if recommendations is not None:
    display(recommendations.query("recommendation == 'REDUCE'").sort_values(['recommended_fleet_delta', 'utilization_pct']).head(20))

In [ ]:
# Rows where utilization is high but recommendation is not BUY. Useful for checking conflict logic.
if recommendations is not None:
    display(recommendations.query("utilization_pct >= 90 and recommendation != 'BUY'").sort_values('utilization_pct', ascending=False))

In [ ]:
# Rows where utilization is low but recommendation is not REDUCE. Useful for checking offsetting positive signals.
if recommendations is not None:
    display(recommendations.query("utilization_pct < 75 and recommendation != 'REDUCE'").sort_values('utilization_pct'))

In [ ]:
# Verify non-positive margin never receives BUY.
if recommendations is not None:
    display(recommendations.query("daily_margin <= 0"))

In [ ]:
# Station-level net fleet movement for planning review.
if recommendations is not None:
    display(recommendations.groupby('station').agg(
        net_delta=('recommended_fleet_delta', 'sum'),
        buy_rows=('recommendation', lambda s: (s == 'BUY').sum()),
        reduce_rows=('recommendation', lambda s: (s == 'REDUCE').sum()),
        avg_utilization=('utilization_pct', 'mean'),
        avg_margin=('daily_margin', 'mean'),
    ).round(2).sort_values('net_delta', ascending=False).head(25))